In [1]:

import sys
import os
import functools
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../../maxtext')))
os.environ["SKIP_JAX_PRECOMPILE"] = "1"

import jax.numpy as jnp
from flax import nnx
import sys
import os
import flax.linen as nn
import logging

import MaxText as mt
from MaxText import pyconfig
from tunix.rl.rollout.vllm_rollout import VllmRollout
from tunix.rl.rollout import base_rollout
import transformers
import jax 
from MaxText.integration.tunix.tunix_adaptor import TunixMaxTextLlama
from tunix.rl import utils
from tunix.models.llama3 import model as llama3_lib


/home/wenxindong_google_com/miniconda3/envs/tunix/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:84: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(
2025-08-13 23:05:19.523193: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1755126319.536006  434880 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1755126319.539742  434880 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:0

INFO 08-13 23:05:25 [__init__.py:241] Automatically detected platform tpu.
WARNING 08-13 23:05:25 [__init__.py:14] 🚨  CAUTION: You are using 'tpu_commons' , which is experimental and NOT intended for production use yet. Please see the README for more details.
INFO 08-13 23:05:25 [__init__.py:16] TPU info: node_name=wenxindong-v6e-8 | tpu_type=v6e-8 | worker_id=0 | num_chips=8 | num_cores_per_chip=1
INFO 08-13 23:05:25 [__init__.py:29] Running vLLM without Pathways. Module pathwaysutils is not imported.


In [2]:
def get_ref_maxtext_model(config):

  #python3 -m MaxText.train MaxText/configs/base.yml base_output_directory=${BASE_OUTPUT_DIRECTORY} dataset_path=${DATASET_PATH} tokenizer_path=assets/tokenizer.gemma load_parameters_path=${CONVERTED_CHECKPOINT} per_device_batch_size=1 run_name=${FINETUNE_RUN_NAME} max_target_length=8192 steps=10 async_checkpointing=false model_name=gemma-2b checkpoint_period=5


  def create_model(config):
    return mt.from_pretrained(config, rngs=nnx.Rngs(params=0, dropout=1))

  abstract_model = nnx.eval_shape(create_model, config=config)
  graphdef, abstract_state = nnx.split(abstract_model)
  print('The abstract NNX state (all leaves are abstract arrays):')
  nnx.display(abstract_state)
  specs = nnx.get_partition_spec(abstract_state)
  mesh = abstract_model.mesh

  # JIT a function that creates the model state with proper sharding from the start.
  # By providing out_shardings, we instruct JAX to produce sharded output directly,
  # avoiding a large intermediate allocation on a single device.
  with nn.logical_axis_rules(config.logical_axis_rules):
    out_shardings = nn.logical_to_mesh_sharding(specs, mesh)

  @functools.partial(jax.jit, out_shardings=out_shardings)
  def create_sharded_state():
      # This will be JIT-compiled. JAX knows the output sharding and can
      # initialize the parameters directly on the target devices in a sharded way.
      model = create_model(config)
      return nnx.state(model)

  with jax.sharding.use_mesh(mesh):
    # Create the model with sharded parameters.
    sharded_state = create_sharded_state()
    model = nnx.merge(graphdef, sharded_state)

    target_for_restore = jax.tree.map(
        lambda v: v.value, sharded_state, is_leaf=lambda n: isinstance(n, nnx.Variable)
    )
    checkpoint = mt.checkpointing.load_params_from_path(
        load_parameters_from_path=config.load_parameters_path,
        abstract_unboxed_params=target_for_restore,
        checkpoint_storage_concurrent_gb=None,
    )
    # checkpoint = ocp.StandardCheckpointer().restore(
    #     "gs://maxtext-gemma/2b/2025-08-05-04-37/0/items", target=target_for_restore
    # )
    if checkpoint:
        nnx.update(model, checkpoint)

    tunix_model = TunixMaxTextLlama(
            base_model=model,
            use_attention_mask=False,  # trust Tunix loss masking
        )

    model_config = llama3_lib.ModelConfig.llama3_1_8b()
    tunix_model.config = model_config

  return tunix_model, mesh, model_config


from MaxText.integration.tunix.tunix_adaptor import TunixMaxTextLlama

config_ref = pyconfig.initialize(
      ["", "../../maxtext/MaxText/configs/base.yml"], #TODO: @mazumdera: why decode.py?
      base_output_directory="gs://dummy_output_dir",  # This is not used in Tunix.
      run_name="test-tunix-maxtext-llama3.1-8b",
      # run_name="test-tunix-maxtext-llama3.1-8b",
      # dataset_path=we use Tunix's dataset
      #TODO: @mazumdera: change this to use checkpoint
      tokenizer_type="tiktoken",
      tokenizer_path="assets/tokenizer_llama3.tiktoken",
      load_parameters_path="gs://maxtext-model-checkpoints/llama3.1-8b/2025-01-23-19-04/scanned/0/items",
      # tokenizer_path="assets/tokenizer.gemma",
      per_device_batch_size=1,
      max_prefill_predict_length=4,
      max_target_length=16,
      steps=10,
      async_checkpointing="false",
      model_name="llama3.1-8b",
      # model_name="gemma-2b",
      checkpoint_period=5,
      skip_jax_distributed_system="true",
      weight_dtype="bfloat16",
      attention="dot_product",
      remat_policy="custom",
      decoder_layer_input="offload",
      query_proj="offload",
      key_proj="offload",
      value_proj="offload",
      opt_type="sgd",
  )

llama_8b, mesh, model_config = get_ref_maxtext_model(config_ref)
# gemma_maxtext_nnx = nnx.bridge.ToNNX(gemma)
# Instead of:
nnx.display(llama_8b)


Updating keys from env and command line: ['run_name', 'model_name', 'load_parameters_path', 'async_checkpointing', 'checkpoint_period', 'weight_dtype', 'remat_policy', 'decoder_layer_input', 'query_proj', 'key_proj', 'value_proj', 'attention', 'base_output_directory', 'tokenizer_path', 'tokenizer_type', 'per_device_batch_size', 'steps', 'skip_jax_distributed_system', 'max_target_length', 'max_prefill_predict_length', 'opt_type']
Running Model: llama3.1-8b
Updating following parameters in config

base_emb_dim: 4096
base_num_query_heads: 32
base_num_kv_heads: 8
base_num_decoder_layers: 32
base_mlp_dim: 14336
head_dim: 128
mlp_activations: ['silu', 'linear']
vocab_size: 128256
enable_dropout: False
logits_via_embedding: False
normalization_layer_epsilon: 1e-05
rope_max_timescale: 500000
decoder_block: llama2
Updating keys from model: ['base_emb_dim', 'base_num_query_heads', 'base_num_kv_heads', 'base_num_decoder_layers', 'base_mlp_dim', 'head_dim', 'mlp_activations', 'vocab_size', 'enable

Not using emergency checkpoint, ignoring local_checkpoint_directory, local_checkpoint_period, use_replicator_service and replicator_backup_interval_minutes
dataset_type set to tfds, will use keys['dataset_path']='' and keys['dataset_name']='c4/en:3.0.1'
Config param activations_in_float32: False
Config param adam_b1: 0.9
Config param adam_b2: 0.95
Config param adam_eps: 1e-08
Config param adam_eps_root: 0.0
Config param adam_weight_decay: 0.1
Config param add_bos: True
Config param add_eos: True
Config param allow_split_physical_axes: False
Config param ar_cache_axis_order: 1,2,0,3
Config param async_checkpointing: False
Config param attention: dot_product
Config param attention_type: global
Config param attn_logits_soft_cap: None
Config param autoregressive_decode_assert: 
Config param base_emb_dim: 4096
Config param base_mlp_dim: 14336
Config param base_moe_mlp_dim: 7168
Config param base_num_decoder_layers: 32
Config param base_num_kv_heads: 8
Config param base_num_query_heads: 32
C

Num_devices: 8, shape (1, 1, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1)
restoring params from gs://maxtext-model-checkpoints/llama3.1-8b/2025-01-23-19-04/scanned/0/items
Creating checkpoint manager with ocdbt=True and zarr3=True
Checkpoint manager created!


In [3]:

# show_hbm_usage = utils.show_hbm_usage

# show_hbm_usage("Before loading model")
# def get_ref_maxtext_model():

#   #TODO: @mazumdera: change this to use Gemma2-2b-it
#   config = pyconfig.initialize(
#       ["", "../../maxtext/MaxText/configs/base.yml"], #TODO: @mazumdera: why decode.py?
#       base_output_directory="gs://dummy_output_dir",  # This is not used in Tunix.
#       run_name="none",
#       tokenizer_path="../../maxtext/assets/tokenizer.gemma",
#       per_device_batch_size=1,
#       max_target_length=1024,
#       steps=10,
#       async_checkpointing="false",
#       model_name="llama3.1-8b", #"llama3.1-8b"
#       checkpoint_period=5, 
#       skip_jax_distributed_system="true",
#       weight_dtype="bfloat16",
#       attention="dot_product"
#   )
  
#   def create_model(config):
#     return mt.from_pretrained(config, rngs=nnx.Rngs(params=0, dropout=1))

#   model = nnx.eval_shape(create_model, config=config)

#   abstract_model = nnx.eval_shape(create_model, config=config)
#   graphdef, abstract_state = nnx.split(abstract_model)
#   print('The abstract NNX state (all leaves are abstract arrays):')
#   nnx.display(abstract_state)

#   @nnx.jit
#   def partial_init(config):
#     model = create_model(config)
#     # nnx.update(model, checkpoint)
#     # shard model
#     state = nnx.state(model)
#     specs = nnx.get_partition_spec(state)
#     state = jax.lax.with_sharding_constraint(state, specs)
#     nnx.update(model, state)
#     return model

#   with jax.sharding.use_mesh(model.mesh), nn.logical_axis_rules(config.logical_axis_rules):
#     model = partial_init(config)
#   print(model)

#   tunix_model = TunixMaxTextLlama(
#         base_model=model,
#         use_attention_mask=False,  # trust Tunix loss masking
#     )
#   mesh  = tunix_model.base.mesh
  
#   tunix_model.to_hf_mappings = lambda *args: {}
#   tunix_model.to_hf_transpose_keys = lambda *args: {}
#   tunix_model.lora_to_hf_mappings = lambda *args: {}

#   # Add these lines to properly get the graph definition and state
#   graphdef, state = nnx.split(tunix_model)
#   tunix_model = nnx.merge(graphdef, state)  # Recreate model in proper NNX format
    
#   return tunix_model, mesh

# model, mesh = get_ref_maxtext_model()

# print(model)
# show_hbm_usage("After loading model")
# nnx.display(nnx.state(model))

In [4]:
model = llama_8b


TOTAL_GENERATION_STEPS = 64
MAX_PROMPT_LENGTH = 64  
TEMPERATURE = 0.9
TOP_P = 1.0
TOP_K = None
cache_config = base_rollout.RolloutConfig(max_tokens_to_generate=TOTAL_GENERATION_STEPS, max_prompt_length=MAX_PROMPT_LENGTH, kv_cache_size=MAX_PROMPT_LENGTH + TOTAL_GENERATION_STEPS + 256, temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K)




In [ ]:
def create_maxtext_to_vllm_mappings():
    """Create mappings for transferring MaxText scanned state to vLLM unscanned state."""
    return {
        # Token embeddings - shard vocab dimension for TP
        'base.token_embedder.embedding': ('embed.embedding', ('model', None)),
        
        # Final layer norm - no sharding needed
        'base.decoder.decoder_norm.scale': ('model.norm.scale', (None,)),
        
        # LM head (logits projection) - shard vocab dimension for TP
        'base.decoder.logits_dense.kernel': ('lm_head', (None, 'model')),
        
        # Layer-specific mappings (scanned -> unscanned)
        # MLP components - shard hidden dimensions for TP
        'base.decoder.layers.mlp.wi_0.kernel': ('model.layers.*.mlp.gate_proj.kernel', (None, 'layer', 'model')),  # gate_proj: (4096, 14336) - shard output
        'base.decoder.layers.mlp.wi_1.kernel': ('model.layers.*.mlp.up_proj.kernel', (None, 'layer', 'model')),    # up_proj: (4096, 14336) - shard output  
        'base.decoder.layers.mlp.wo.kernel': ('model.layers.*.mlp.down_proj.kernel', ('model', 'layer', None)),    # down_proj: (14336, 4096) - shard input
        
        # Layer norms - no sharding needed
        'base.decoder.layers.pre_self_attention_layer_norm.scale': ('model.layers.*.input_layernorm.scale', (None, 'layer')),
        'base.decoder.layers.post_self_attention_layer_norm.scale': ('model.layers.*.post_attention_layernorm.scale', (None, 'layer')),
        
        # Attention components - shard head dimensions for TP
        'base.decoder.layers.self_attention.query.kernel': ('model.layers.*.self_attn.q_proj.kernel', (None, 'layer', 'model', None)),  # q_proj: shard num_heads
        'base.decoder.layers.self_attention.key.kernel': ('model.layers.*.self_attn.k_proj.kernel', (None, 'layer', 'model', None)),    # k_proj: shard num_kv_heads
        'base.decoder.layers.self_attention.value.kernel': ('model.layers.*.self_attn.v_proj.kernel', (None, 'layer', 'model', None)),  # v_proj: shard num_kv_heads
        'base.decoder.layers.self_attention.out.kernel': ('model.layers.*.self_attn.o_proj.kernel', ('model', 'layer', None, None)),    # o_proj: shard input heads
    }




In [ ]:
mappings = create_maxtext_to_vllm_mappings()

transpose_keys = {
    # MLP transposes (after layer extraction)
    'wo.kernel': (1, 0),  # down_proj: (14336, 4096) - transpose needed
    
    # Attention output transpose (after layer extraction) 
    'out.kernel': (1, 2, 0),  # o_proj: (32, 128, 4096) -> (32, 128, 4096) - reorder dimensions
}


model.to_hf_mappings = create_maxtext_to_vllm_mappings
model.to_hf_transpose_keys = lambda *args: transpose_keys
model.lora_to_hf_mappings = lambda *args: None  # No LoRA

In [7]:
model_tokenizer = transformers.AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B")

In [ ]:
TOTAL_TPU_TO_USE = 8
MESH = [(1, TOTAL_TPU_TO_USE), ("fsdp", "tp")]  # YY

mesh =   model_mesh = jax.make_mesh(*MESH, devices=jax.devices()[:TOTAL_TPU_TO_USE])


In [ ]:
from tunix.generate.vllm_sampler import VllmSampler, MappingConfig

sampler = VllmSampler(
    mesh=mesh,
    tokenizer=model_tokenizer,
    max_model_len=1024,
    model_version="meta-llama/Llama-3.1-8b",
    mapping_config=MappingConfig(),
    hbm_utilization=0.3,  # Adjust based on your TPU memory
)


AttributeError: 'dict' object has no attribute 'lora_config'

In [9]:
rollout = VllmRollout(model=model,tokenizer=model_tokenizer,cache_config_or_size=64, mesh=mesh,lora_config=None,model_version="meta-llama/Llama-3.1-8B")


INFO 08-13 23:06:06 [utils.py:326] non-default args: {'model': 'meta-llama/Llama-3.1-8B', 'max_model_len': 64, 'tensor_parallel_size': 8, 'gpu_memory_utilization': 0.3, 'disable_log_stats': True}
WARNING 08-13 23:06:06 [__init__.py:516] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.


INFO 08-13 23:06:16 [__init__.py:707] Resolved architecture: LlamaForCausalLM
INFO 08-13 23:06:16 [__init__.py:1731] Using max model len 64


2025-08-13 23:06:17,064	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 08-13 23:06:17 [__init__.py:2031] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 08-13 23:06:17 [tpu_jax.py:147] The model dtype is not properly set for JAX backend. Overwriting it to jnp.bfloat16


/home/wenxindong_google_com/miniconda3/envs/tunix/lib/python3.12/site-packages/torch_xla/__init__.py:258: UserWarning: `tensorflow` can conflict with `torch-xla`. Prefer `tensorflow-cpu` when using PyTorch/XLA. To silence this warning, `pip uninstall -y tensorflow && pip install tensorflow-cpu`. If you are in a notebook environment such as Colab or Kaggle, restart your notebook runtime afterwards.
  warnings.warn(


INFO 08-13 23:06:18 [tpu_jax.py:182] Force using UniProcExecutor for JAX on single host.
INFO 08-13 23:06:19 [core.py:72] Initializing a V1 LLM engine (v0.1.dev8444+g094bed7da.d20250811) with config: model='meta-llama/Llama-3.1-8B', speculative_config=None, tokenizer='meta-llama/Llama-3.1-8B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=False, dtype=<class 'jax.numpy.bfloat16'>, max_seq_len=64, download_dir=None, load_format=auto, tensor_parallel_size=8, pipeline_parallel_size=1, disable_custom_all_reduce=True, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_config=None, decoding_config=DecodingConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_backend=''), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collect_detailed_traces=None), seed=0, served_

In [11]:
from tunix.rl.rollout.base_rollout import RolloutConfig



rollout.generate(["hello world", "how are you?"], rollout_config= RolloutConfig(n=1))

Processed prompts: 100%|██████████| 2/2 [00:00<00:00,  5.39it/s, est. speed input: 26.97 toks/s, output: 318.23 toks/s]


RolloutOutput(text=['!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!', '!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!'], logits=None, tokens=Array([[     0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0, 128001, 128001, 128001, 128001],
       [     0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0, 